# Indexing 3: refine and build the adaptive basis (10 degree mosaicity)

1. Refine the pbp map: every peak is assigned a position along the beam
   (its diffraction origin), and the candidate grains of every voxel are
   refined against the peaks passing through it.
2. Segment the refined map into grains (flood fill on the best orientation) and sort the
   candidates of every voxel by completeness (unique peaks relative to the
   best voxel of the same grain).
3. Build the adaptive basis: all candidate orientations of all voxels, thinned so that
   no two orientations are closer than `theta_deg`. This is the orientation grid
   used in the texture tomography.

In [ ]:
%load_ext autoreload
%autoreload 2
%matplotlib inline

import sys
from pathlib import Path

# make the repository importable, whatever the working directory is
REPO = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "simtools").is_dir())
sys.path.insert(0, str(REPO))

import os

import numpy as np
import matplotlib.pyplot as plt
import ImageD11.unitcell
import ImageD11.sinograms.point_by_point as pbp
from scipy.spatial.transform import Rotation as R
from diffractom import Grid
from simtools import dummy, io, orientations, paths, pbp_maps, plot

plot.dark()

In [ ]:
SAMPLE = "domains_mosaicity_10p0deg"
savefile = os.path.join(paths.process_dir(SAMPLE), "peaks_2d")

colf = io.read_peaks(paths.peaks_path(SAMPLE))
ypositions, ystep, ymin, y0 = dummy.add_pbp_columns(colf)
dset = dummy.load_dset(savefile + "_dummy_dset.pkl")
dset.y0 = y0
pbpmap = pbp.PBPMap(savefile + "_pbpmap.h5")

In [ ]:
pars = colf.parameters.parameters
ucell = ImageD11.unitcell.unitcell(
    [pars["cell__a"], pars["cell__b"], pars["cell__c"], pars["cell_alpha"], pars["cell_beta"], pars["cell_gamma"]],
    pars["cell_lattice_[P,A,B,C,I,F,R]"],
)
ucell.makerings(colf.ds.max())

In [ ]:
multi_channel = pbp_maps.MultiChannelMap(pbpmap, ucell)
print(f"{multi_channel.nchannels} candidate grains (channels) per voxel at most")

## Refinement

In [ ]:
dset.refmapfile = savefile + "_pbpmap.h5"  # pbp map input
dset.refpeaksfile = savefile + "_icolfile_refine.h5"  # peaks with diffraction origins
dset.refoutfile = savefile + "_pbpmap_refine.h5"  # refined pbp map
dset.refmanfile = savefile + "_pbprefiner.h5"  # the refiner itself

eta_wedge = 6
etacut = np.abs(np.sin(np.radians(eta_wedge)))
ifrac = colf.sum_intensity.min() / colf.sum_intensity.max()  # use all peaks

refiner = pbp.PBPRefine(
    dset,
    phase_name=None,
    hkl_tol_origins=0.15,
    hkl_tol_refine=0.08,
    hkl_tol_refine_merged=0.08,
    ds_tol=0.012,
    etacut=etacut,
    ifrac=ifrac,
    forref=None,
    y0=y0,
    min_grain_npks=10,
)
refiner.setmap(pbpmap)
refiner.setpeaks(colf, del_existing=True, prompt_del=False)

The sample mask: voxels whose reconstructed peak intensity is above `manual_threshold`.

In [ ]:
refiner.setmask(manual_threshold=0.65, doplot=True, use_icolf=True)

Diffraction origins are computed from the best candidate of every voxel.

In [ ]:
refiner.singlemap = multi_channel.ubi[:, :, 0]
refiner.get_origins(guess_speed=False, save_peaks_after=True)

xpos = refiner.icolf.xpos_refined[refiner.icolf.isel] if hasattr(refiner.icolf, "isel") else refiner.icolf.xpos_refined
fig, ax = plt.subplots(1, 1, figsize=(12, 6))
ax.hist(xpos, bins=300)
ax.set_xlabel("diffraction origin along the beam, x (um)")
ax.set_ylabel("peaks")
plot.despine(ax)
plt.show()

In [ ]:
output_filename = savefile + "_refined_pbpmap.h5"
if os.path.isfile(output_filename):
    os.remove(output_filename)
refiner.run_refine(points_step_space=None, npoints=None, output_filename=output_filename, use_cluster=False, pythonpath=None)

In [ ]:
refined_pbpmap = pbp.PBPMap(refiner.refinedmap_filename)
refined = pbp_maps.MultiChannelMap(refined_pbpmap, ucell)

fig, ax = plt.subplots(figsize=(12, 4))
ax.hist(refined_pbpmap.nuniq, bins=np.arange(0.5, np.max(refined_pbpmap.nuniq) + 0.51, 1))
ax.set_xlabel("unique diffraction spots per candidate grain")
ax.set_ylabel("candidates")
plot.despine(ax)
plt.show()

## Grain segmentation and completeness

In [ ]:
segmap = pbp_maps.segment_grains(
    refined,
    refiner.mask,
    local_disorientation_tolerance=5,  # deg, between neighbouring voxels
    max_grains=999,
    min_grain_size=5,  # voxels
    seed=0,
)
refined.normalize_per_grain(segmap)
refined.sort()  # by completeness, then total peaks
print(f"{int(np.nanmax(segmap))} grains")

In [ ]:
fig, ax = plt.subplots(1, 3, figsize=(24, 8))
ax[0].imshow(segmap, cmap="rainbow")
ax[0].set_title("grains")
for a, (field, title) in zip(ax[1:], [(refined.nuniq[:, :, 0], "unique peaks"), (refined.completeness[:, :, 0], "completeness")]):
    _s = field.copy()
    _s[~refiner.mask] = np.nan
    im = a.imshow(_s, cmap="rainbow")
    a.set_title(title)
    fig.colorbar(im, ax=a, fraction=0.046, pad=0.04)
plot.despine(ax)
plt.tight_layout()
plt.show()

_s = refined.rgb[:, :, 0, 2].copy()
_s[~refiner.mask] = np.nan
fig, ax = plt.subplots(1, 1, figsize=(9, 9))
ax.pcolormesh(dset.sx_grid, dset.sy_grid, _s)
ax.set_aspect("equal")
ax.set_title("IPF-z, best candidate")
ax.set_xlabel("x (um)")
ax.set_ylabel("y (um)")
plot.despine(ax)
plt.show()

## Build the adaptive basis

All candidate orientations of all voxels, in the order of the sorted map, are
thinned greedily: an orientation is kept unless it is within `theta_deg` of one
already kept (`diffractom.Grid.prune_close_orientations`).

In [ ]:
theta_deg = 0.4

u = refined.u.reshape(-1, 3, 3)
u = u[np.any(u != 0, axis=(1, 2))]  # drop empty channels
print(f"{len(u)} candidate orientations")

grid = Grid.from_rotation_matrices(u, sigma=np.deg2rad(theta_deg))  # sigma is not used for pruning
grid.prune_close_orientations(theta_deg=theta_deg)
basis = R.concatenate(grid.rotations_at_level(0)).as_matrix()
print(f"adaptive basis: {len(basis)} orientations")

os.makedirs(os.path.dirname(paths.basis_path(SAMPLE)), exist_ok=True)
np.save(paths.basis_path(SAMPLE), basis)
print("saved", paths.basis_path(SAMPLE))

## Comparison with the ground truth
The misorientation from each ground-truth mesh element to the closest basis orientation.

In [ ]:
gt = io.read_ground_truth(SAMPLE)
idx = np.random.default_rng(0).choice(len(gt["orientation"]), 5000, replace=False)
mis = orientations.min_misorientation_deg(gt["orientation"][idx], basis)

fig, ax = plt.subplots(1, 1, figsize=(12, 6))
ax.hist(mis, bins=100, range=(0, max(5, np.percentile(mis, 99))))
ax.axvline(np.median(mis), c="w", label=f"median {np.median(mis):.2f} deg")
ax.set_xlabel("misorientation to the closest basis orientation (deg)")
ax.set_ylabel("ground-truth mesh elements")
ax.legend()
plot.despine(ax)
plt.show()